# Heat Exchanger Network Design of a Four-Stream Problem
This notebook demonstrates the public heat exchanger network design workflow on a compact four-stream problem. It uses the problem-owned `design` accessor for direct use and the `PinchWorkspace` workflow dispatcher for app-style variant execution.

## Imports

In [ ]:
import warnings

import pandas as pd
from OpenPinch import PinchProblem, PinchWorkspace
from OpenPinch.lib import HeatExchangerKind, HeatExchangerNetworkLabel, ZT
from OpenPinch.lib.schemas.io import TargetInput

NetworkLabel = HeatExchangerNetworkLabel

warnings.filterwarnings(
    "ignore",
    message="__array__ implementation doesn't accept a copy keyword.*",
)

## Four-stream case
The stream and utility data are the converted four-stream heat exchanger network synthesis example used by the regression suite. The heat exchanger network design grid uses three approach temperatures, one derivative threshold, and one stage count so the notebook can show the top three networks without sending Couenne through a broad teaching sweep.

In [ ]:
four_stream_case = {
    "zone_tree": {
        "name": "Site",
        "type": ZT.S.value,
        "children": [
            {"name": "Process A", "type": ZT.P.value},
        ],
    },
    "options": {
        "COST_EXP": 1.0,
        "FIXED_COST": 5500.0,
        "VARIABLE_COST": 150.0,
        "HENS_APPROACH_TEMPERATURES": [10.0, 14.0, 18.0],
        "HENS_DERIVATIVE_THRESHOLDS": [0.5],
        "HENS_STAGE_SELECTION": [3],
        "HENS_BEST_SOLUTIONS_TO_SAVE": 3,
        "HENS_MAX_PARALLEL": 1,
        "HENS_OUTPUT_FORMATS": [],
        "HENS_RUN_ID": "four-stream-hen-demo",
    },
    "streams": [
        {
            "zone": "Site/Process A",
            "name": "Raw Milk",
            "t_supply": {"value": 650.0, "unit": "K"},
            "t_target": {"value": 370.0, "unit": "K"},
            "heat_flow": {"value": 2800.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
        {
            "zone": "Site/Process A",
            "name": "HT Flash",
            "t_supply": {"value": 590.0, "unit": "K"},
            "t_target": {"value": 370.0, "unit": "K"},
            "heat_flow": {"value": 4400.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
        {
            "zone": "Site/Process A",
            "name": "Milk Concentrate",
            "t_supply": {"value": 410.0, "unit": "K"},
            "t_target": {"value": 650.0, "unit": "K"},
            "heat_flow": {"value": 3600.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
        {
            "zone": "Site/Process A",
            "name": "CIP Water",
            "t_supply": {"value": 350.0, "unit": "K"},
            "t_target": {"value": 500.0, "unit": "K"},
            "heat_flow": {"value": 1950.0, "unit": "kW"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
        },
    ],
    "utilities": [
        {
            "name": "HPS",
            "type": "Hot",
            "t_supply": {"value": 680.0, "unit": "K"},
            "t_target": {"value": 680.0, "unit": "K"},
            "htc": {"value": 5.0, "unit": "kW/m^2/K"},
            "price": {"value": 80.0, "unit": "$/MWh"},
        },
        {
            "name": "CW",
            "type": "Cold",
            "t_supply": {"value": 300.0, "unit": "K"},
            "t_target": {"value": 320.0, "unit": "K"},
            "htc": {"value": 1.0, "unit": "kW/m^2/K"},
            "price": {"value": 15.0, "unit": "$/MWh"},
        },
    ],
}

## Validate and prepare a problem
`TargetInput` gives an explicit schema boundary before the live `PinchProblem` is created.

In [ ]:
target_input = TargetInput.model_validate(four_stream_case)
problem = PinchProblem(
    source=target_input,
    project_name="four_stream_hen_design_service_demo",
)
validated = problem.validate()

pd.DataFrame(
    [
        {
            "project": problem.project_name,
            "streams": len(validated.streams),
            "utilities": len(validated.utilities or []),
            "approach_temperatures": validated.options["HENS_APPROACH_TEMPERATURES"],
            "derivative_thresholds": validated.options["HENS_DERIVATIVE_THRESHOLDS"],
            "stage_selection": validated.options["HENS_STAGE_SELECTION"],
        }
    ]
)

## Run the design workflow
The public service entry point is owned by the problem: `problem.design.heat_exchanger_network_synthesis()`. By default this call invokes the local solver-backed synthesis executor. Heat exchanger network configuration belongs in the case options loaded above; runtime options are reserved for target execution state such as `state_id`. Running this cell requires the `openpinch[synthesis]` Python extra and external solver binaries; `idaes get-extensions` installs Couenne and Ipopt into the IDAES extension directory, which OpenPinch will discover automatically.

In [ ]:
design = problem.design.heat_exchanger_network_synthesis()
network = design.network

pd.DataFrame(
    [
        {
            "run_id": design.run_id,
            "accepted_method": design.method,
            "solver_name": design.solver_name,
            "solver_status": design.solver_status,
            "stage_count": design.stage_count,
            "task_count": len(design.task_outcomes),
            "exchanger_count": len(network.exchangers),
            "cached_on_problem": problem.results.design == design,
        }
    ]
)

## Inspect the workflow manifest
The result carries serializable workflow metadata alongside the accepted network.

In [ ]:
manifest = design.manifest

manifest_frame = pd.DataFrame(
    [
        {
            "run_id": manifest.run_id,
            "approach_temperatures": list(manifest.approach_temperatures),
            "derivative_thresholds": list(manifest.derivative_thresholds),
            "stage_selection": list(manifest.stage_selection),
            "task_ids": len(manifest.task_ids),
            "export_formats": list(manifest.export_formats),
        }
    ]
)

task_frame = pd.DataFrame(
    [
        {
            "method": outcome.task.method,
            "status": outcome.status,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
        }
        for outcome in design.task_outcomes
    ]
)

display(manifest_frame)
task_frame.groupby(["method", "status"], as_index=False).size()

## Top network grid designs
Successful network candidates are ranked by objective value. The Plotly grid diagrams below show the top three candidates from the accepted method family. Hover over exchanger markers to inspect match duty, area, and stream pairing.


In [ ]:
network_candidates = [
    outcome
    for outcome in design.task_outcomes
    if outcome.status == "success"
    and outcome.network is not None
    and outcome.objective_value is not None
]
accepted_method_candidates = [
    outcome for outcome in network_candidates if outcome.task.method == design.method
]
top_network_outcomes = sorted(
    accepted_method_candidates or network_candidates,
    key=lambda outcome: outcome.objective_value,
)[:3]

pd.DataFrame(
    [
        {
            "rank": rank,
            "method": outcome.task.method,
            "task_id": outcome.task.task_id,
            "approach_temperature_K": outcome.task.approach_temperature,
            "derivative_threshold": outcome.task.derivative_threshold,
            "stage_count": outcome.task.stage_count,
            "objective_value": outcome.objective_value,
            "recovery_duty_kW": outcome.network.total(NetworkLabel.RECOVERY_DUTY),
            "hot_utility_duty_kW": outcome.network.total(NetworkLabel.HOT_UTILITY_DUTY),
            "cold_utility_duty_kW": outcome.network.total(NetworkLabel.COLD_UTILITY_DUTY),
        }
        for rank, outcome in enumerate(top_network_outcomes, start=1)
    ]
)

In [ ]:
grid_diagrams = [
    design.grid_diagram(solution_rank=rank)
    for rank in range(1, min(3, len(top_network_outcomes)) + 1)
]

for rank, diagram in enumerate(grid_diagrams, start=1):
    diagram.fig.update_layout(title_text=f"Rank {rank} heat exchanger network grid")
    display(diagram.fig)


The returned diagram object wraps a Plotly figure. `diagram.save("network.png")` writes a static image through Kaleido, and `diagram.save("network.html")` writes an interactive Plotly document.


## Inspect the accepted network
The accepted `HeatExchangerNetwork` exposes ordered exchanger records plus stable labelled accessors for totals and stream-to-stream matches.

In [ ]:
exchanger_frame = pd.DataFrame(
    [
        {
            "id": exchanger.exchanger_id,
            "kind": exchanger.kind.value,
            "source": exchanger.source_stream,
            "sink": exchanger.sink_stream,
            "stage": exchanger.stage,
            "duty_kW": exchanger.duty,
            "area": exchanger.area,
            "active": exchanger.active,
            "match_allowed": exchanger.match_allowed,
            "source_outlet_K": exchanger.source_outlet_temperature,
            "sink_outlet_K": exchanger.sink_outlet_temperature,
        }
        for exchanger in network.exchangers
    ]
)

exchanger_frame

In [ ]:
first_recovery = next(
    exchanger
    for exchanger in network.exchangers
    if exchanger.kind is HeatExchangerKind.RECOVERY
)

pd.DataFrame(
    [
        {"metric": "total recovery duty", "value": network.total(NetworkLabel.RECOVERY_DUTY)},
        {"metric": "total hot utility duty", "value": network.total(NetworkLabel.HOT_UTILITY_DUTY)},
        {"metric": "total cold utility duty", "value": network.total(NetworkLabel.COLD_UTILITY_DUTY)},
        {
            "metric": "first recovery match duty",
            "value": network.labelled_value(
                NetworkLabel.RECOVERY_DUTY,
                source_stream=first_recovery.source_stream,
                sink_stream=first_recovery.sink_stream,
                stage=first_recovery.stage,
            ),
        },
    ]
)

## Run through `PinchWorkspace`
Workspace callers can dispatch the same design workflow by name and then read the design payload from the solved case.

In [ ]:
workspace = PinchWorkspace(
    source=four_stream_case,
    project_name="four_stream_hen_workspace_demo",
)
view = workspace.solve_variant(
    "baseline",
    workflow="heat_exchanger_network_synthesis",
)
workspace_design = workspace.case("baseline").results.design

pd.DataFrame(
    [
        {
            "variant": view.variant_name,
            "status": view.status,
            "support_level": view.support_level,
            "workspace_variant": workspace_design.workspace_variant,
            "run_id": workspace_design.run_id,
            "method": workspace_design.method,
            "network_exchangers": len(workspace_design.network.exchangers),
        }
    ]
)